# DistilBERT + features (roadmap step 5)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohaaa825/Skolfie/blob/main/notebooks/transformer.ipynb)

Fine-tunes DistilBERT together with the step 4 style and GPT-2 features ("Model A") and scores
it on the same held-out splits as the earlier models:

* `transformer`: DistilBERT's reading of the text + the style and GPT-2 features in one network
* `transformer-text` (optional): DistilBERT on the text alone, to see what the features add

**Use a GPU runtime:** Runtime → Change runtime type → T4 GPU. Then **Runtime → Run all**.

The GPT-2 features are already cached in `My Drive/skolfie/cache/` from step 4, so they load in
seconds. Fine-tuning on the full train split takes roughly an hour on a T4 (one pass over
176k texts), plus a few minutes to score the test splits. Keep the tab open: unlike the
feature cache, a training run that is cut off has to start again. For a quicker first look,
set `MAX_TRAIN = 60000`.

Reports go to `My Drive/skolfie/reports/<model>/`, trained models to `My Drive/skolfie/models/`.

## 1. Settings

In [ ]:
DATA_DIR = "/content/drive/MyDrive/skolfie/data/processed"
REPORTS_DIR = "/content/drive/MyDrive/skolfie/reports"
CACHE_DIR = "/content/drive/MyDrive/skolfie/cache"
MODELS_DIR = "/content/drive/MyDrive/skolfie/models"

REPO = "mohaaa825/Skolfie"
BRANCH = "main"

MODELS = ["transformer"]  # add "transformer-text" to measure DistilBERT without the features
EPOCHS = 1
BATCH_SIZE = 16           # lower to 8 if you get a CUDA out-of-memory error
MAX_TRAIN = None          # e.g. 60000 for a quicker run; None = full train split
MAX_EVAL = None           # e.g. 20000 to cap each test split; None = everything

## 2. Check the GPU, mount Drive, clone and install

In [ ]:
import os
import subprocess

import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("No GPU: switch to a T4 GPU runtime (Runtime -> Change runtime type).")

from google.colab import drive

drive.mount("/content/drive")
for d in (REPORTS_DIR, CACHE_DIR, MODELS_DIR):
    os.makedirs(d, exist_ok=True)
assert os.path.exists(f"{DATA_DIR}/corpus.parquet"), "Run build_corpus.ipynb first."
assert os.path.exists(f"{DATA_DIR}/splits.parquet"), "Run evaluate.ipynb (step 3) first."


def secret(name):
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.isdir("/content/Skolfie/.git"):
    subprocess.run(["git", "-C", "/content/Skolfie", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/Skolfie"],
                   check=True)

%cd /content/Skolfie
!git log --oneline -1
!pip install -q -e ".[features]"

## 3. Train and score

DistilBERT (about 260 MB) and GPT-2 are downloaded from Hugging Face the first time; set the
`HF_TOKEN` Colab secret if downloads are rate-limited. Progress lines show the training speed
and the time left.

In [ ]:
import sys

hf_token = secret("HF_TOKEN")
env = dict(os.environ, **({"HF_TOKEN": hf_token} if hf_token else {}))


def run(cmd):
    """Run a command and stream its output into the notebook as it happens."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            bufsize=1, env=env)
    for line in proc.stdout:
        sys.stdout.write(line)
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


extra = []
if MAX_TRAIN:
    extra += ["--max-train", str(MAX_TRAIN)]
if MAX_EVAL:
    extra += ["--max-eval", str(MAX_EVAL)]

for model in MODELS:
    print(f"\n========== {model} ==========", flush=True)
    run(["python", "-u", "scripts/evaluate.py",
         "--corpus", f"{DATA_DIR}/corpus.parquet",
         "--splits", f"{DATA_DIR}/splits.parquet",
         "--model", model,
         "--out-dir", f"{REPORTS_DIR}/{model}",
         "--cache-dir", CACHE_DIR,
         "--epochs", str(EPOCHS),
         "--batch-size", str(BATCH_SIZE),
         "--save-model", f"{MODELS_DIR}/{model}.joblib",
         "--save-predictions", *extra])

## 4. Compare with the earlier models

In [ ]:
import glob

reports = sorted(glob.glob(f"{REPORTS_DIR}/*/report.json"))
out = subprocess.run(["python", "scripts/evaluate.py", "--compare", *reports],
                     check=True, capture_output=True, text=True).stdout
print(out)
with open(f"{REPORTS_DIR}/comparison.md", "w") as f:
    f.write(out)

Reports are in `My Drive/skolfie/reports/`. Tell Claude in the project thread when this has
finished; it reads the report cards from Drive directly.